# Batch Deployment

Batch inference is the most common way of deploying machine learning models. This lesson introduces various strategies for deploying models using batch including Spark. In addition, we will show how to enable optimizations for Delta tables.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

* Load a logged Model Registry model using `pyfunc`.
* Compute predictions using `pyfunc` APIs.
* Perform batch inference using Feature Engineering's `score_batch` method.
* Materialize predictions into inference tables (Delta Lake).
* Perform common write optimizations like liquid clustering, predictive optimization to maximize data skipping and on inference tables.

# Requirements

Please review the following requirements before starting the lesson:
* To run this notebook, you need to use one of the following Databricks runtime(s): **{{supported_dbrs}}**

🚨 **Prerequisites:**

* **Feature Engineering** and **Feature Store** are not focus of this lesson. This course expect that you already know these topics. If not, you can check the **Data Preparation for Machine Learning** course.
* Model development with MLflow is not in the scope of this course. If you need to refresh your knowledge about model tracking and logging, you can check the **Machine Learning Model Development** course.

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [0]:
import os

# 1. Configurar as credenciais do Kaggle no ambiente do cluster
os.environ['KAGGLE_USERNAME'] = "fabienecasamento" #"SEU_USUARIO_AQUI"
with open('/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/chave_kaggle.txt', 'r') as leitura_chave:
    kaggle_key = leitura_chave.read().strip()
# Extrair os dados antes do ';|' na variável kaggle_key
kaggle_key_before_delimiter = kaggle_key.split(';')[0]


os.environ['KAGGLE_KEY'] = kaggle_key_before_delimiter #"SUA_CHAVE_AQUI" # token databricks

In [0]:
# 2. Instalar a biblioteca do Kaggle via comando de terminal no cluster
# (O caractere '!' permite rodar comandos shell diretamente do notebook)
!pip install -q kaggle

# 3. Criar uma pasta local temporária no driver e baixar o dataset do link informado
!mkdir -p /tmp/kaggle_data
!kaggle datasets download -d blastchar/telco-customer-churn -p /tmp/kaggle_data --unzip

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


I0000 00:00:1780616722.660043   12819 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1780616722.798252   12819 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100%|████████████████████████████████████████| 172k/172k [00:00<00:00, 2.86MB/s]



In [0]:
# 4. Copiar o arquivo do /tmp para o Workspace (serverless requer /Workspace ou Volumes)
import shutil
import os

workspace_dir = "/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/kaggle_data"
os.makedirs(workspace_dir, exist_ok=True)
shutil.copy("/tmp/kaggle_data/WA_Fn-UseC_-Telco-Customer-Churn.csv", f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 5. Abrir e ler o arquivo utilizando o PySpark
# O arquivo baixado do Kaggle adota o nome padrão original da IBM ("WA_Fn-UseC_-...")
dataset_path = f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv"

telco_df = spark.read.csv(
    dataset_path, 
    inferSchema=True, 
    header=True, 
    multiLine=True, 
    escape='"'
)

# 6. Visualizar o DataFrame carregado com sucesso no Spark
display(telco_df.show(8))

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
display(telco_df.toPandas().tail(8))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
8456-QDAVC,Male,0,No,No,19,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),78.7,1495.1,No
7750-EYXWZ,Female,0,No,No,12,No,No phone service,DSL,No,Yes,Yes,Yes,Yes,Yes,One year,No,Electronic check,60.65,743.3,No
2569-WGERO,Female,0,No,No,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.4,No
6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.8,1990.5,No
2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.2,7362.9,No
4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.6,346.45,No
8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.4,306.6,Yes
3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


In [0]:
telco_df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
from pyspark.sql.functions import col


# dataset path
#dataset_p_telco = f"{DA.paths.datasets}/telco/telco-customer-churn.csv"

# features to use
primary_key = "customerID"
response = "Churn"
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] # Keeping numerical only for simplicity and demo purposes

#telco_df = telco_df.withColumn("TotalCharges", col("TotalCharges").cast('double'))\
#    .withColumn("SeniorCitizen", col("SeniorCitizen").cast('double'))\
#    .withColumn("Tenure", col("tenure").cast('double'))\
#    .na.drop(how='any')

    
from pyspark.sql.functions import expr

telco_df = telco_df.withColumn("TotalCharges", expr("try_cast(TotalCharges as double)"))\
    .withColumn("SeniorCitizen", expr("try_cast(SeniorCitizen as double)"))\
    .withColumn("Tenure", expr("try_cast(tenure as double)"))\
    .na.drop(how='any')

# Split with 80 percent of the data in train_df and 20 percent of the data in test_df
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

# Separate features and ground-truth
features_df = train_df.select(primary_key, *features)
response_df = train_df.select(primary_key, response)

# review the features dataset


In [0]:
telco_df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: double (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Tenure: double (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
display(features_df.show(3))

+----------+-------------+------+--------------+------------+
|customerID|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|
+----------+-------------+------+--------------+------------+
|0002-ORFBO|          0.0|   9.0|          65.6|       593.3|
|0003-MKNFE|          0.0|   9.0|          59.9|       542.4|
|0004-TLHLJ|          0.0|   4.0|          73.9|      280.85|
+----------+-------------+------+--------------+------------+
only showing top 3 rows


In [0]:
display(response_df.show(3))

+----------+-----+
|customerID|Churn|
+----------+-----+
|0002-ORFBO|   No|
|0003-MKNFE|   No|
|0004-TLHLJ|  Yes|
+----------+-----+
only showing top 3 rows


# Batch Deployment - Without Feature Store

This demo will cover two main batch deployment methods. The first method is deploying models without a feature table. For the second method, we will use a feature table to train the model and later use the feature table for inference.

## Setup Model Registry with UC

Before we start model deployment, we need to fit and register a model. In this demo, **we will log models to Unity Catalog**, which means first we need to setup the **MLflow Model Registry URI**.

In [0]:
import mlflow


# Point to UC model registry
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

# helper function that we will use for getting latest version of a model
def get_latest_model_version(model_name):
    """Helper function to get latest model version"""
    model_version_infos = client.search_model_versions(f"name = '%s'" % model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

In [0]:
# List available catalogs and schemas in Unity Catalog
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
display(catalogs)



_1
samples
system
workspace


In [0]:
# Choose a catalog and list schemas
#selected_catalog = catalogs[2]  # or set manually
#print(selected_catalog)
#schemas = [row.schema for row in spark.sql(f"SHOW SCHEMAS IN {selected_catalog}").collect()]
#display(schemas)


In [0]:

# Set catalog_name and schema_name variables
catalog_name = catalogs[2] 
#schema_name = schemas[0]  # or set manually

schema_name = 'information_schema' # sem premissão
print(f"{catalog_name}.{schema_name}")

workspace.information_schema


In [0]:
# Train a sklearn Decision Tree Classification model
from sklearn.tree import DecisionTreeClassifier
from mlflow.models import infer_signature


# Set catalog_name and schema_name variables
catalog_name = catalogs[2] 
#schema_name = schemas[0]  # or set manually

schema_name = 'default'
print(f"{catalog_name}.{schema_name}")

# Covert data to pandas dataframes
X_train_pdf = features_df.drop(primary_key).toPandas()
Y_train_pdf = response_df.drop(primary_key).toPandas()
clf = DecisionTreeClassifier(max_depth=3, random_state=42)

# Use 3-level namespace for model name
model_name = f"{catalog_name}.{schema_name}.ml_model"

with mlflow.start_run(run_name="Model-Batch-Deployment-Demo") as mlflow_run:

    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True
    )

    clf.fit(X_train_pdf, Y_train_pdf)

    # Log model and push to registry
    signature = infer_signature(X_train_pdf, Y_train_pdf)
    mlflow.sklearn.log_model(
        clf,
        artifact_path="decision_tree",
        signature=signature,
        registered_model_name=model_name
    )

    # Set model alias (i.e. Baseline)
    client.set_registered_model_alias(model_name, "Baseline", get_latest_model_version(model_name))

workspace.default


2026/06/05 00:11:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/2770952798348874/models/m-c11a3c13d7144addb776e3d89ea08f3d?o=7474657872577658
Successfully registered model 'workspace.default.ml_model'.


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.ml_model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/ml_model/version/1?o=7474657872577658


![image_1780618385106.png](./image_1780618385106.png "image_1780618385106.png")

![image_1780618435965.png](./image_1780618435965.png "image_1780618435965.png")

In [0]:
#página 36/38
# clicando no model-batch-deployment-demo
# no mlflow

![image_1780618515646.png](./image_1780618515646.png "image_1780618515646.png")

![image_1780618570382.png](./image_1780618570382.png "image_1780618570382.png")

In [0]:
#conda.yaml

#channels:
#- conda-forge
#dependencies:
#- python=3.12.3
#- pip<=25.0.1
#- pip:
#  - mlflow==3.8.1
#  - cloudpickle==3.0.0
#  - pandas==2.2.3
#  - psutil==5.9.0
#  - scikit-learn==1.6.1
#name: mlflow-env


In [0]:
#python_env.yaml
#python: 3.12.3
#build_dependencies:
#- pip==25.0.1
#- setuptools==78.1.1
#- wheel==0.45.1
#dependencies:
#- -r requirements.txt




In [0]:
#requirements.txt
#mlflow==3.8.1
#cloudpickle==3.0.0
#pandas==2.2.3
#psutil==5.9.0
#scikit-learn==1.6.1

In [0]:
# página 46